# 03 — Prototypage de l'optimisation budgétaire

**Formulation** (multiple-choice knapsack) — pour le client *i* et l'action *k* :
`gain_ik = uplift_ik × valeur_conversion − coût_k`, variables binaires `x_ik`.

maximiser Σ gain_ik·x_ik  sous  Σ_k x_ik ≤ 1 ; Σ coût_k·x_ik ≤ budget ; (option) Σ_i x_ik ≤ cap_k.

**Questions de recherche**
1. Combien rapporte la politique optimisée vs ciblage naïf / aléatoire, selon le budget ?
2. Quand la programmation linéaire apporte-t-elle vraiment quelque chose par rapport à un tri glouton ?

In [1]:
import sys, warnings

sys.path.insert(0, "../src")            # pas d'installation nécessaire
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import load_config

cfg = load_config()
ARMS, CTRL = cfg["arms"], cfg["control_label"]
pd.set_option("display.float_format", lambda v: f"{v:.5f}")

from train import ensure_scores
from optimization import optimize_allocation, naive_top_uplift, random_policy, evaluate_policy_model_based, net_gain_matrix

scores = ensure_scores(cfg)
U = scores[[f"uplift_{a}" for a in ARMS]].rename(columns=lambda c: c.removeprefix("uplift_"))
VALUE, COSTS = cfg["business"]["value_per_conversion"], cfg["business"]["cost_per_contact"]
print(len(U), "clients | uplift moyen par bras :", U.mean().round(5).to_dict())

INFO source=hillstrom | fit=35840 val=8960 test=19200
INFO [val] s_learner       mean_qini=0.000080
INFO [val] t_learner       mean_qini=0.000308
INFO [val] x_learner       mean_qini=0.000010
INFO [val] response_model  mean_qini=0.000399
INFO best=t_learner | test mean_qini=0.000248 (response_model=0.000271)


19200 clients | uplift moyen par bras : {'Mens E-Mail': 0.00687, 'Womens E-Mail': 0.00316}


## 1. Gain attendu selon le budget (gain net estimé par le modèle)

In [2]:
rows = []
for B in [50, 100, 200, 400, 600, 1000, 2000]:
    opt, _ = optimize_allocation(U, VALUE, COSTS, B, CTRL)
    for name, pol in [("LP", opt), ("naïf top-uplift", naive_top_uplift(U, COSTS, B, CTRL)), ("aléatoire", random_policy(U, COSTS, B, CTRL, 0))]:
        s = evaluate_policy_model_based(pol, U, VALUE, COSTS, CTRL)
        rows.append(dict(budget=B, policy=name, contacts=s["n_contacted"], net_gain=s["expected_net_gain"]))
sweep = pd.DataFrame(rows); piv = sweep.pivot(index="budget", columns="policy", values="net_gain"); display(piv)
piv.plot(marker="o", title="Gain net attendu vs budget"); plt.ylabel("EUR"); plt.show()
assert (piv["LP"] >= piv["naïf top-uplift"] - 1e-6).all(), "le LP ne doit jamais être battu par le glouton"

policy,LP,aléatoire,naïf top-uplift
budget,,,
50,145.23000,23.35000,144.97000
100,275.34000,45.86000,275.34000
200,513.04000,99.06000,513.04000
400,916.12000,204.47000,915.94000
600,1247.19000,303.71000,1247.04000
1000,1759.98000,510.34000,1759.88000
2000,2216.69000,971.73000,2187.78000


## 2. Quand le LP gagne : coûts différents + capacité limitée sur un bras

In [ ]:
costs2 = {ARMS[0]: 0.10, ARMS[1]: 0.25}            # email Femmes plus coûteux (ex. catalogue + offre)
caps = {ARMS[1]: int(len(U) * 0.05)}               # stock limité : 5 % de la base max sur le bras 2
B = 600
lp, s_lp = optimize_allocation(U, VALUE, costs2, B, CTRL, max_per_arm=caps)
nv = naive_top_uplift(U, costs2, B, CTRL)

# le naïf ignore les capacités : on applique la contrainte a posteriori en retirant l'excédent
over = nv[nv == ARMS[1]].index[caps[ARMS[1]]:]
nv_capped = nv.copy(); nv_capped.loc[over] = CTRL
for name, pol in [("LP (contraint)", lp), ("naïf + coupe a posteriori", nv_capped)]:
    print(name, evaluate_policy_model_based(pol, U, VALUE, costs2, CTRL))

LP (contraint) {'status': 'evaluated', 'n_contacted': 6000, 'contacts_per_arm': {'Mens E-Mail': 6000}, 'total_cost': 600.0, 'expected_incremental_conversions': 60.83, 'expected_net_gain': 1225.02}
naïf + coupe a posteriori {'status': 'evaluated', 'n_contacted': 5542, 'contacts_per_arm': {'Mens E-Mail': 5237, 'Womens E-Mail': 305}, 'total_cost': 599.95, 'expected_incremental_conversions': 57.68, 'expected_net_gain': 1130.57}


## 3. Où part le budget ? (profil des clients contactés)

In [4]:
pol, _ = optimize_allocation(U, VALUE, COSTS, 600, CTRL)
prof = scores.assign(action=pol.values)
display(prof.groupby("action")[["recency", "history", "mens", "womens", "newbie"]].mean())
display(prof.action.value_counts().to_frame("clients"))

,recency,history,mens,womens,newbie
action,,,,,
Mens E-Mail,5.97546,419.42937,0.60858,0.61141,0.43344
No E-Mail,5.74159,153.91656,0.52023,0.52295,0.51955
Womens E-Mail,1.52381,638.32702,0.51488,0.95238,1.00000


,clients
action,
No E-Mail,13200
Mens E-Mail,5664
Womens E-Mail,336
